In [ ]:
"""
Foundation Model Classifiers for Bulk RNA-seq Binary Classification
====================================================================
Supports:
  1. BulkFormer  — pretrained on 500K+ bulk RNA-seq profiles (BEST for your use case)
  2. Geneformer  — pretrained on 104M single-cell transcriptomes (cancer-tuned variant)

Both expose the same .fit() / .predict() / .predict_proba() API so they drop
directly into your existing cfm() + StratifiedKFold evaluation loop.

Install dependencies:
    pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
    pip install transformers datasets anndata scanpy tqdm scikit-learn imbalanced-learn
    pip install git+https://github.com/KangBoming/BulkFormer.git   # BulkFormer
    pip install git+https://huggingface.co/ctheodoris/Geneformer    # Geneformer
"""


In [1]:

# ─────────────────────────────────────────────────────────────────────────────
# 0. Shared imports
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
from tqdm import tqdm


# ─────────────────────────────────────────────────────────────────────────────
# Evaluation helper  (same as your existing cfm())
# ─────────────────────────────────────────────────────────────────────────────
def cfm(y_test, y_pred, y_prob):
    auc = roc_auc_score(y_test, y_prob)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    acc = accuracy_score(y_test, y_pred)
    recall = tp / (tp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    specificity = tn / (tn + fp)
    sensitivity = tp / (tp + fn)
    f1 = round((2 * recall * precision) / (recall + precision + 1e-9), 3)
    return np.array([acc, recall, precision, specificity, sensitivity, f1, auc])




In [2]:

class BulkFormerClassifier:
    """
    Fine-tunes BulkFormer for binary classification on bulk RNA-seq data.

    BulkFormer is natively pretrained on bulk RNA-seq (not single-cell),
    making it the best foundation model choice for your GSE68086 data.

    Architecture:
        BulkFormer encoder (GNN + Performer) → CLS embedding → 2-class head

    Usage:
        clf = BulkFormerClassifier(gene_names=list(X_train.columns), device='cuda')
        clf.fit(X_train.values, y_train_array)
        y_pred = clf.predict(X_test.values)
        y_prob = clf.predict_proba(X_test.values)[:, 1]
    """

    def __init__(
        self,
        gene_names: list,
        num_classes: int = 2,
        epochs: int = 20,
        batch_size: int = 16,
        lr: float = 1e-4,
        freeze_encoder: bool = False,     # True = linear probe, False = full fine-tune
        use_smote: bool = True,
        device: str = "cuda" if torch.cuda.is_available() else "cpu",
        model_path: str = None,           # path to BulkFormer pretrained weights
    ):
        self.gene_names   = gene_names
        self.num_classes  = num_classes
        self.epochs       = epochs
        self.batch_size   = batch_size
        self.lr           = lr
        self.freeze_encoder = freeze_encoder
        self.use_smote    = use_smote
        self.device       = device
        self.model_path   = model_path
        self.model        = None
        self.le           = LabelEncoder()

    # ------------------------------------------------------------------
    def _load_bulkformer(self):
        """Load BulkFormer encoder + attach a classification head."""
        try:
            from bulkformer.model import BulkFormerModel
            from bulkformer.config import BulkFormerConfig
        except ImportError:
            raise ImportError(
                "BulkFormer not installed. Run:\n"
                "  pip install git+https://github.com/KangBoming/BulkFormer.git"
            )

        config = BulkFormerConfig(
            gene_vocab=self.gene_names,
            num_labels=self.num_classes,
        )
        if self.model_path:
            model = BulkFormerModel.from_pretrained(self.model_path, config=config)
        else:
            # Downloads pretrained weights from HuggingFace automatically
            model = BulkFormerModel.from_pretrained("KangBoming/BulkFormer", config=config)

        if self.freeze_encoder:
            # Linear probe: freeze all encoder layers, train only the head
            for name, param in model.named_parameters():
                if "classifier" not in name:
                    param.requires_grad = False

        return model.to(self.device)

    # ------------------------------------------------------------------
    def _prepare_bulkformer_input(self, X: np.ndarray):
        """
        BulkFormer expects:
          - expression: (N, G) float tensor of log-normalized counts
          - gene_ids:   (N, G) int tensor of gene vocabulary indices
        """
        try:
            from bulkformer.tokenizer import BulkFormerTokenizer
        except ImportError:
            raise ImportError("BulkFormer package not found.")

        tokenizer = BulkFormerTokenizer(gene_vocab=self.gene_names)
        encodings = tokenizer(
            gene_names=self.gene_names,
            expression=X,         # already log-normalized from your DESeq2 pipeline
            return_tensors="pt",
        )
        return encodings

    # ------------------------------------------------------------------
    def fit(self, X: np.ndarray, y):
        y_enc = self.le.fit_transform(y)

        # Optional SMOTE for class imbalance
        if self.use_smote:
            sm = SMOTE(k_neighbors=3, random_state=42)
            X, y_enc = sm.fit_resample(X, y_enc)

        self.model = self._load_bulkformer()
        optimizer  = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, self.model.parameters()),
            lr=self.lr, weight_decay=1e-4
        )
        scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.epochs)
        loss_fn    = nn.CrossEntropyLoss()

        self.model.train()
        N = len(X)
        for epoch in range(self.epochs):
            idx     = np.random.permutation(N)
            X_shuf  = X[idx]
            y_shuf  = y_enc[idx]
            ep_loss = 0.0

            for start in range(0, N, self.batch_size):
                X_batch = X_shuf[start : start + self.batch_size]
                y_batch = torch.tensor(
                    y_shuf[start : start + self.batch_size], dtype=torch.long
                ).to(self.device)

                inputs  = self._prepare_bulkformer_input(X_batch)
                inputs  = {k: v.to(self.device) for k, v in inputs.items()}

                optimizer.zero_grad()
                outputs = self.model(**inputs)
                logits  = outputs.logits          # (B, num_classes)
                loss    = loss_fn(logits, y_batch)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                ep_loss += loss.item()

            scheduler.step()
            print(f"Epoch {epoch+1}/{self.epochs}  loss={ep_loss/max(1,N//self.batch_size):.4f}")

    # ------------------------------------------------------------------
    @torch.no_grad()
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        self.model.eval()
        probs = []
        for start in range(0, len(X), self.batch_size):
            X_batch = X[start : start + self.batch_size]
            inputs  = self._prepare_bulkformer_input(X_batch)
            inputs  = {k: v.to(self.device) for k, v in inputs.items()}
            logits  = self.model(**inputs).logits
            probs.append(torch.softmax(logits, dim=-1).cpu().numpy())
        return np.vstack(probs)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.le.classes_[np.argmax(self.predict_proba(X), axis=1)]



In [3]:

# =============================================================================
# ██████╗ ███████╗███╗   ██╗███████╗███████╗ ██████╗ ██████╗ ███╗   ███╗███████╗██████╗
# ██╔════╝ ██╔════╝████╗  ██║██╔════╝██╔════╝██╔═══██╗██╔══██╗████╗ ████║██╔════╝██╔══██╗
# ██║  ███╗█████╗  ██╔██╗ ██║█████╗  █████╗  ██║   ██║██████╔╝██╔████╔██║█████╗  ██████╔╝
# ██║   ██║██╔══╝  ██║╚██╗██║██╔══╝  ██╔══╝  ██║   ██║██╔══██╗██║╚██╔╝██║██╔══╝  ██╔══██╗
# ╚██████╔╝███████╗██║ ╚████║███████╗██║     ╚██████╔╝██║  ██║██║ ╚═╝ ██║███████╗██║  ██║
# =============================================================================

class GeneformerClassifier:
    """
    Fine-tunes Geneformer (V2-104M cancer-tuned) for binary bulk RNA-seq classification.

    Strategy: pseudo-cell approach — treat each bulk sample as a single "cell".
    Rank genes by expression per sample → Geneformer tokenizer → BERT-style fine-tune.

    Two modes:
        mode='finetune'    — full fine-tuning with classification head (recommended)
        mode='embedding'   — extract CLS embeddings, train a lightweight head on top
                             (faster, better when n is very small)

    Usage:
        clf = GeneformerClassifier(gene_names=list(X_train.columns), mode='finetune')
        clf.fit(X_train.values, y_train_array)
        y_pred = clf.predict(X_test.values)
        y_prob = clf.predict_proba(X_test.values)[:, 1]
    """

    GENEFORMER_HF_ID = "ctheodoris/Geneformer"
    # Cancer-tuned variant (pretrained on 14M cancer cells via continual learning):
    CANCER_HF_ID     = "ctheodoris/Geneformer"   # use submodel Geneformer-V2-104M_CLcancer

    def __init__(
        self,
        gene_names: list,
        num_classes: int = 2,
        mode: str = "finetune",        # 'finetune' or 'embedding'
        epochs: int = 10,
        batch_size: int = 8,
        lr: float = 2e-5,
        use_smote: bool = True,
        max_input_size: int = 2048,    # token budget per sample
        use_cancer_model: bool = True,
        device: str = "cuda" if torch.cuda.is_available() else "cpu",
    ):
        self.gene_names       = gene_names
        self.num_classes      = num_classes
        self.mode             = mode
        self.epochs           = epochs
        self.batch_size       = batch_size
        self.lr               = lr
        self.use_smote        = use_smote
        self.max_input_size   = max_input_size
        self.use_cancer_model = use_cancer_model
        self.device           = device
        self.model            = None
        self.tokenizer        = None
        self.head             = None   # used in 'embedding' mode
        self.le               = LabelEncoder()

    # ------------------------------------------------------------------
    def _rank_genes(self, x: np.ndarray) -> list:
        """
        Convert a bulk expression vector to a ranked gene list (Geneformer format).
        Genes are sorted highest → lowest expression; truncated at max_input_size.
        Returns list of Ensembl IDs (genes not in Geneformer vocab are dropped).
        """
        ranked_idx = np.argsort(x)[::-1]
        ranked_genes = [self.gene_names[i] for i in ranked_idx
                        if self.gene_names[i] in self.gf_vocab]
        return ranked_genes[: self.max_input_size]

    # ------------------------------------------------------------------
    def _load_geneformer(self):
        try:
            from transformers import AutoTokenizer, AutoModel, AutoConfig
            from transformers import BertForSequenceClassification
        except ImportError:
            raise ImportError("Install transformers: pip install transformers")

        hf_id = self.CANCER_HF_ID if self.use_cancer_model else self.GENEFORMER_HF_ID
        print(f"Loading Geneformer from {hf_id} ...")

        if self.mode == "finetune":
            from transformers import BertConfig, BertForSequenceClassification
            model = BertForSequenceClassification.from_pretrained(
                hf_id,
                num_labels=self.num_classes,
                ignore_mismatched_sizes=True,
            )
        else:
            # Embedding mode: plain BERT encoder
            from transformers import BertModel
            model = BertModel.from_pretrained(hf_id)

        return model.to(self.device)

    # ------------------------------------------------------------------
    def _build_vocab(self):
        """Load Geneformer's gene-to-token mapping."""
        try:
            from geneformer import TranscriptomeTokenizer
            tk = TranscriptomeTokenizer()
            self.gf_vocab = tk.gene_token_dict        # {ensembl_id: token_id}
        except ImportError:
            # Fallback: build identity vocab from provided gene names
            print("Warning: geneformer package not found; using identity vocab.")
            self.gf_vocab = {g: i for i, g in enumerate(self.gene_names)}

    # ------------------------------------------------------------------
    def _tokenize_batch(self, X: np.ndarray) -> dict:
        """
        Tokenize a batch of bulk RNA-seq samples for Geneformer.
        Returns dict with 'input_ids' and 'attention_mask' as torch tensors.
        """
        all_input_ids = []
        max_len = 0

        for i in range(len(X)):
            ranked = self._rank_genes(X[i])
            token_ids = [self.gf_vocab.get(g, 0) for g in ranked]
            all_input_ids.append(token_ids)
            max_len = max(max_len, len(token_ids))

        # Pad sequences to max_len in this batch
        pad_id = 0
        input_ids_padded  = []
        attention_masks   = []

        for ids in all_input_ids:
            pad_len = max_len - len(ids)
            input_ids_padded.append(ids + [pad_id] * pad_len)
            attention_masks.append([1] * len(ids) + [0] * pad_len)

        return {
            "input_ids":      torch.tensor(input_ids_padded, dtype=torch.long).to(self.device),
            "attention_mask": torch.tensor(attention_masks,  dtype=torch.long).to(self.device),
        }

    # ------------------------------------------------------------------
    def fit(self, X: np.ndarray, y):
        y_enc = self.le.fit_transform(y)

        # SMOTE on raw features before tokenizing
        if self.use_smote:
            sm = SMOTE(k_neighbors=3, random_state=42)
            X, y_enc = sm.fit_resample(X, y_enc)

        self._build_vocab()
        self.model = self._load_geneformer()

        # ── Embedding mode: extract features first, then train a lightweight head ──
        if self.mode == "embedding":
            print("Extracting Geneformer embeddings ...")
            embeddings = self._extract_embeddings(X)
            self._train_head(embeddings, y_enc)
            return

        # ── Fine-tune mode ────────────────────────────────────────────────────────
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.epochs)
        loss_fn   = nn.CrossEntropyLoss()
        N         = len(X)

        self.model.train()
        for epoch in range(self.epochs):
            idx    = np.random.permutation(N)
            X_shuf = X[idx]
            y_shuf = y_enc[idx]
            ep_loss = 0.0

            for start in tqdm(range(0, N, self.batch_size),
                              desc=f"Epoch {epoch+1}/{self.epochs}", leave=False):
                X_batch = X_shuf[start : start + self.batch_size]
                y_batch = torch.tensor(
                    y_shuf[start : start + self.batch_size], dtype=torch.long
                ).to(self.device)

                inputs  = self._tokenize_batch(X_batch)
                outputs = self.model(**inputs, labels=y_batch)
                loss    = outputs.loss

                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                ep_loss += loss.item()

            scheduler.step()
            print(f"Epoch {epoch+1}/{self.epochs}  loss={ep_loss/max(1,N//self.batch_size):.4f}")

    # ------------------------------------------------------------------
    @torch.no_grad()
    def _extract_embeddings(self, X: np.ndarray) -> np.ndarray:
        """Extract mean-pooled CLS token embeddings from the Geneformer encoder."""
        self.model.eval()
        all_embs = []
        for start in tqdm(range(0, len(X), self.batch_size), desc="Extracting embeddings"):
            inputs  = self._tokenize_batch(X[start : start + self.batch_size])
            outputs = self.model(**inputs)
            # Use [CLS] token (position 0) as sample representation
            cls_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embs.append(cls_emb)
        return np.vstack(all_embs)

    # ------------------------------------------------------------------
    def _train_head(self, embeddings: np.ndarray, y_enc: np.ndarray):
        """Train a 2-layer MLP head on top of frozen Geneformer embeddings."""
        in_dim = embeddings.shape[1]
        self.head = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, self.num_classes),
        ).to(self.device)

        opt     = torch.optim.Adam(self.head.parameters(), lr=1e-3)
        loss_fn = nn.CrossEntropyLoss()
        X_t     = torch.tensor(embeddings, dtype=torch.float32)
        y_t     = torch.tensor(y_enc, dtype=torch.long)
        ds      = TensorDataset(X_t, y_t)
        loader  = DataLoader(ds, batch_size=32, shuffle=True)

        self.head.train()
        for epoch in range(50):
            for xb, yb in loader:
                xb, yb = xb.to(self.device), yb.to(self.device)
                opt.zero_grad()
                loss_fn(self.head(xb), yb).backward()
                opt.step()

    # ------------------------------------------------------------------
    @torch.no_grad()
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        self.model.eval()
        probs = []

        if self.mode == "embedding":
            embs   = self._extract_embeddings(X)
            X_t    = torch.tensor(embs, dtype=torch.float32).to(self.device)
            self.head.eval()
            logits = self.head(X_t)
            return torch.softmax(logits, dim=-1).cpu().numpy()

        for start in range(0, len(X), self.batch_size):
            inputs  = self._tokenize_batch(X[start : start + self.batch_size])
            logits  = self.model(**inputs).logits
            probs.append(torch.softmax(logits, dim=-1).cpu().numpy())
        return np.vstack(probs)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.le.classes_[np.argmax(self.predict_proba(X), axis=1)]


# =============================================================================
# EVALUATION LOOP — drop-in replacement for your existing SVM loop
# =============================================================================

def evaluate_foundation_model(clf, X_train, X_test, y_train, y_test, label="Model"):
    """
    Trains clf on (X_train, y_train) and evaluates on (X_test, y_test).
    Returns a dict matching your df_result column format.
    Uses the same cfm() function and threshold selection strategy as your SVM code.
    """
    print(f"\n{'='*60}")
    print(f"  Evaluating: {label}")
    print(f"{'='*60}")

    clf.fit(X_train, y_train)

    # ── Training-set optimal threshold (same logic as your SVM code) ──
    y_prob_train = clf.predict_proba(X_train)[:, 1]
    fpr_tr, tpr_tr, thresh_tr = roc_curve(y_train, y_prob_train)
    best_threshold = thresh_tr[np.argmax(tpr_tr - fpr_tr)]
    print(f"  Optimal threshold (Youden): {best_threshold:.4f}")

    # ── Test set evaluation ──
    y_prob = clf.predict_proba(X_test)[:, 1]
    y_pred_default  = clf.predict(X_test)                              # default 0.5 threshold
    y_pred_optimal  = (y_prob >= best_threshold).astype(int)          # Youden threshold
    y_pred_optimal  = clf.le.classes_[y_pred_optimal]

    metrics_default = cfm(y_test, y_pred_default, y_prob)
    metrics_optimal = cfm(y_test, y_pred_optimal, y_prob)

    def fmt(arr):
        names = ['ACC', 'Recall', 'Precision', 'Sp', 'Sn', 'F1', 'AUC']
        return {n: f"{v*100:.2f}%" for n, v in zip(names, arr)}

    print(f"\n  Default threshold (0.5):")
    for k, v in fmt(metrics_default).items():
        print(f"    {k}: {v}")

    print(f"\n  Optimal threshold ({best_threshold:.3f}):")
    for k, v in fmt(metrics_optimal).items():
        print(f"    {k}: {v}")

    return {
        "model":            label,
        "default_thresh":   fmt(metrics_default),
        "optimal_thresh":   fmt(metrics_optimal),
        "best_threshold":   best_threshold,
    }



In [4]:

# =============================================================================
# CROSS-VALIDATION LOOP (mirrors your existing StratifiedKFold structure)
# =============================================================================

def cv_evaluate(clf_class, clf_kwargs, X, y, n_splits=5, label="Model"):
    """
    Stratified K-fold CV for any foundation model classifier.
    Returns mean ± std metrics across folds.
    """
    skf     = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    le      = LabelEncoder()
    y_enc   = le.fit_transform(y)
    results = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y_enc)):
        print(f"\n  Fold {fold+1}/{n_splits}")
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        clf = clf_class(**clf_kwargs)
        clf.fit(X_tr, y_tr)

        y_prob = clf.predict_proba(X_va)[:, 1]
        y_pred = clf.predict(X_va)
        results.append(cfm(y_va, y_pred, y_prob))

    results_arr = np.array(results)
    names = ['ACC', 'Recall', 'Precision', 'Sp', 'Sn', 'F1', 'AUC']
    print(f"\n  {'─'*50}")
    print(f"  {label} — {n_splits}-Fold CV Summary")
    print(f"  {'─'*50}")
    for i, name in enumerate(names):
        m, s = results_arr[:, i].mean(), results_arr[:, i].std()
        print(f"  {name:<12} {m*100:.2f}% ± {s*100:.2f}%")

    return results_arr



In [9]:

import ast

X_train = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\ranked rf\X_train_rF_DEG_train_lung.csv", low_memory=False)
X_test  = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\ranked rf\X_test_rF_DEG_train_lung.csv",  low_memory=False)
y_train = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\ranked rf\y_train_rF_DEG_train.csv",      low_memory=False)
y_test  = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\ranked rf\y_test_rF_DEG_train.csv",       low_memory=False)

# Apply your SFS-selected gene set (same as end of your existing script)
# df_result2 = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\sfs\SVM_DEG_on_training(test_selected)_25_5_lungAUC.csv")
# best_genes = ast.literal_eval(df_result2.iloc[36]["Gene"])
# X_train = X_train[best_genes]
# X_test  = X_test[best_genes]

gene_names = list(X_train.columns)   # Ensembl IDs (ENSG...) required by Geneformer
X_tr = X_train.values.astype(np.float32)
X_te = X_test.values.astype(np.float32)
y_tr = y_train["cancer_type"].values
y_te = y_test["cancer_type"].values


In [10]:

# ── Option A: BulkFormer fine-tune (RECOMMENDED) ──────────────────────────
print("\n" + "="*60)
print("  BulkFormer — bulk RNA-seq native foundation model")
print("="*60)

bulkformer_clf = BulkFormerClassifier(
    gene_names   = gene_names,
    epochs       = 20,
    batch_size   = 16,
    lr           = 1e-4,
    freeze_encoder = False,   # full fine-tune; set True for linear probe
    use_smote    = True,
    device       = "cuda" if torch.cuda.is_available() else "cpu",
)
results_bf = evaluate_foundation_model(
    bulkformer_clf, X_tr, X_te, y_tr, y_te, label="BulkFormer"
)



  BulkFormer — bulk RNA-seq native foundation model

  Evaluating: BulkFormer


ImportError: BulkFormer not installed. Run:
  pip install git+https://github.com/KangBoming/BulkFormer.git

In [ ]:

# ── Option B: Geneformer embedding mode (fast, good for small n) ──────────
print("\n" + "="*60)
print("  Geneformer (cancer-tuned) — embedding + MLP head")
print("="*60)

geneformer_emb = GeneformerClassifier(
    gene_names       = gene_names,
    mode             = "embedding",   # 'finetune' for full fine-tune
    epochs           = 10,
    batch_size       = 8,
    lr               = 2e-5,
    use_cancer_model = True,          # uses Geneformer-V2-104M_CLcancer
    use_smote        = True,
    device           = "cuda" if torch.cuda.is_available() else "cpu",
)
results_gf = evaluate_foundation_model(
    geneformer_emb, X_tr, X_te, y_tr, y_te, label="Geneformer (embedding)"
)


In [ ]:

# ── Option C: Geneformer full fine-tune ───────────────────────────────────
geneformer_ft = GeneformerClassifier(
    gene_names       = gene_names,
    mode             = "finetune",
    epochs           = 10,
    batch_size       = 8,
    lr               = 2e-5,
    use_cancer_model = True,
    use_smote        = True,
    device           = "cuda",
)
results_gf_ft = evaluate_foundation_model(
    geneformer_ft, X_tr, X_te, y_tr, y_te, label="Geneformer (fine-tune)"
)


In [ ]:

# ── Cross-validation comparison ───────────────────────────────────────────
X_all = np.vstack([X_tr, X_te])
y_all = np.concatenate([y_tr, y_te])

cv_evaluate(
    GeneformerClassifier,
    dict(gene_names=gene_names, mode="embedding", epochs=10,
            batch_size=8, use_cancer_model=True, use_smote=True),
    X_all, y_all, n_splits=5, label="Geneformer (embedding)",
)